# Reproducing *Geometric Dynamics Across Recurrent Vision Models*

Luo, Alvarez & Konkle, CCN 2026.

This notebook regenerates **every table and figure in the paper** from the
saved activations. It needs no GPU, no TensorFlow, and no model code — just
the activation tensors (see the README for the download link) and the
`repgeo` package in this repo.

| Paper item | Section below |
|---|---|
| Table 1 — models & accuracy | §1 |
| Table 2 + Figure 1 — local/global dynamics | §2 |
| Figure 2 — decision-stage prototype distances | §3 |
| Figure 3 — cross-model RSA & MDS | §4 |
| Figure 4 (supplement) — ConvRNN t=12→t=17 | §5 |

Each cell prints the numbers reported in the paper so you can check the
reproduction as you go. Run top to bottom.

In [ ]:
# --- Setup ---------------------------------------------------------------
# Recommended one-time install (from the repo root):   pip install -e .
# After that, `import repgeo` works no matter where Jupyter was launched.
# The fallback below also lets this notebook run without installing,
# as long as it stays inside the repo's notebooks/ folder.
try:
    import repgeo
except ImportError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve().parent))
    import repgeo

# --- Point these at your downloaded activations ----------------------------
# ACTIVATIONS = "../activations"      # directory with one subfolder per model
# SUFFIX      = "imagenetval_100x50"  # dataset tag embedded in every filename

# for self-testing
ACTIVATIONS = "/teamspace/studios/this_studio/reprGeo_output"
SUFFIX      = "imagenetval_100*50"   # local files use the old * naming

import torch
from repgeo import load_features, sanity_check

features, labels = load_features(ACTIVATIONS, suffix=SUFFIX)
sanity_check(features, labels)   # checks every tensor has 5000 rows / 100 classes

## §1 — Table 1: models and accuracy

Two accuracy measures per model and processing stage (baseline → after
recurrence):

* **Top-1 (logits)** — standard ImageNet accuracy from the model's own
  classifier (`source="logits"`).
* **NC Penult. Top-1** — nearest-centroid accuracy on the penultimate layer:
  each image is assigned to the nearest category prototype in cosine
  distance (`source="penult"`). A classifier-independent separability
  measure comparable across all models.

Expected (paper Table 1):

| Model | logits base→after | NC penult base→after |
|---|---|---|
| LRM3 | 57.4 → 59.0 | 80.6 → 83.2 |
| LRA3 | 57.7 → 58.5 | 80.9 → 82.7 |
| BL | 51.1 → 57.0 | 76.0 → 83.4 |
| CORnet-RT | – → 56.2 | 52.5 → 75.9 |
| ConvRNN | 68.0 → 70.9 | 90.0 → 90.1 |

In [ ]:
from repgeo import accuracy_table

print("Standard ImageNet Top-1/5 (logits argmax):")
display(accuracy_table(features, labels, source="logits"))

print("Nearest-centroid Top-1/5 on penultimate features:")
display(accuracy_table(features, labels, source="penult"))

**Why CORnet-RT is analyzed at the penultimate layer.** The paper notes
that CORnet-RT's *logits* at the baseline step are not a meaningful decision
readout — nearest-centroid separability there is only **43.6%** — whereas
ConvRNN's baseline logits are already well separated (**82.0%**). We confirm
both numbers below; this is why CORnet-RT uses the penultimate layer in the
main analyses while the other recurrent models use logits.

In [ ]:
from repgeo import nearest_centroid_accuracy

labels_t = torch.as_tensor(labels)
classes = torch.unique(labels_t, sorted=True).tolist()
for m in ["CORnet-RT", "ConvRNN"]:
    top1, _ = nearest_centroid_accuracy(features[m]["logits"]["baseline"], labels_t, classes)
    print(f"{m:10s} baseline-logits nearest-centroid Top-1 = {top1:.1f}%")

## §2 — Table 2 & Figure 1: local and global dynamics

The headline result. For each recurrent model we contrast the baseline and
after-recurrence stages (CORnet-RT on penultimate, the rest on logits):

* **Local** — change in exemplar-to-prototype cosine distance
  (negative = compaction, exemplars tighten toward their prototype).
* **Global** — prototype shift, change in between-prototype separation
  (centered), and how well the prototype RDM is preserved.

### Table 2

Expected: LRM3 −0.031 / −6.52 / 0.017 / +0.21 / 0.955; LRA3 −0.021 / −4.46 /
0.010 / +0.18 / 0.973; BL +0.014 / +6.58 / 0.028 / +3.90 / 0.977; CORnet-RT
−0.100 / −38.80 / 0.441 / +3.38 / 0.804; ConvRNN +0.023 / +11.50 / 0.018 /
+2.13 / 0.978.

In [ ]:
from repgeo import geometry_summary_table

geometry_summary_table(features, labels).round(3)

### Figure 1 — local/global composite

Top row: local cluster-size-change KDE per model (red dashed = mean, black =
zero). Bottom row: prototype shifts in the baseline-PCA space (gray = baseline,
red = after). CORnet-RT uses the penultimate layer (`rep_overrides`).

In [ ]:
from repgeo import local_global_composite, plotting as P

# Analysis first: compute the composite's inputs (deltas + prototype-shift PCA),
# then hand them to the plotting function (which does no analysis itself).
composite = local_global_composite(
    features, labels, rep="logits", rep_overrides={"CORnet-RT": "penult"})
P.plot_local_global_composite(composite)

### Significance tests

* **Local** — linear mixed-effects model on the per-exemplar distance change,
  random intercepts by category. The paper reports all five models
  significant at *p* < .001.
* **Global** — permutation test (10,000 iterations) swapping the
  baseline/after label per exemplar; also *p* < .001 for all models.

The permutation test is the slow step (~a minute at 10k). Drop `n_perm` to
preview, or raise it back to 10000 to match the paper.

In [ ]:
from repgeo import run_local_lme_tests, run_global_permutation_tests

print("Local (mixed-effects, distance_diff ~ 1 + (1|category)):")
display(run_local_lme_tests(features, labels))

print("Global (between-prototype separation, permutation test):")
display(run_global_permutation_tests(features, labels, n_perm=10000))

## §3 — Figure 2: learned decision-stage structure

Distribution of all 4,950 pairwise cosine distances between the 100 category
prototypes at the **logits** layer. A distribution concentrated near 1.0 means
prototypes are near-orthogonal (maximally separated). LRM/LRA and the standard
ResNets show this; AlexNet, B, VGG-16, and the other recurrent models do not.

Solid line = baseline, dotted = after recurrence. CLIP is omitted (no logits).

In [ ]:
from repgeo import compute_cluster_sizes_and_rdms, between_prototype_summary

cluster_log, between_log = compute_cluster_sizes_and_rdms(features, labels, rep="logits")
P.plot_separation_ridge(between_log)

The in-text means ± SD (e.g. LRM 0.958±0.124 → 0.959±0.115; AlexNet
0.805±0.371; CORnet-RT 0.397±0.200 → 0.807±0.375; ResNet-101 0.991±0.061)
come straight from these distributions:

In [ ]:
between_prototype_summary(between_log)

## §4 — Figure 3: cross-model representational similarity

For each model/stage we build a 100×100 prototype distance matrix and
correlate the upper triangles across all model pairs (Spearman ρ). The 2×2
composite shows the cross-model RSA heatmaps (top) and MDS embeddings (bottom)
at the penultimate (left) and logits (right) layers. Models converge at the
feature stage but split into families at the decision stage.

In [ ]:
from repgeo import rsa_mds_composite, build_prototype_rdms, plotting as P

_, between_pen = compute_cluster_sizes_and_rdms(features, labels, rep="penult")
rdms_log = build_prototype_rdms(features, labels, rep="logits")
rdms_pen = build_prototype_rdms(features, labels, rep="penult")

# Analysis first: cross-model RSA matrices + 2-D MDS embeddings; then render.
rsa_mds = rsa_mds_composite(between_log, between_pen, rdms_log, rdms_pen)
P.plot_rsa_mds_composite(rsa_mds)
print("logits-MDS stress:", round(rsa_mds['mds_log'][3], 3),
      "  penult-MDS stress:", round(rsa_mds['mds_pen'][3], 3))

In [ ]:
# Interactive 3-D MDS (drag to rotate); needs plotly. Build coords, then plot.
# from repgeo import build_mds_coords
# names3, coords3, _, stress3 = build_mds_coords(rdms_log, n_components=3)
# P.plot_model_mds_3d(names3, coords3, "Model MDS 3-D — LOGITS", stress=stress3);

## §5 — Figure 4 (supplement): ConvRNN input-on vs. final state

ConvRNN stops receiving the image after t=12. To isolate *input-independent*
recurrent refinement, the supplement compares **t=12 (last input-driven step)
→ t=17 (final)**, which needs the extra `timgon` activation saved by
`extract_convrnn.py --image-off 12`.

Four panels: (a) cluster-size change, (b) k-NN category preservation, (c)
prototype-shift PCA, (d) prototype-shift histogram.

Expected: mean cluster size +0.046 (26.8%), mean prototype shift 0.183,
between-prototype distances +14.6%, prototype RDM ρ = 0.899.

In [ ]:
from repgeo import load_convrnn_states, convrnn_supplement, pair_geometry_metrics

X, labels_c = load_convrnn_states(ACTIVATIONS, SUFFIX, states=("timgon", "tlast"), rep="logits")
print(pair_geometry_metrics(X["timgon"], X["tlast"], labels_c))

# Analysis first (deltas, k-NN curves, prototype-shift PCA + histogram), then plot.
suppl = convrnn_supplement(X["timgon"], X["tlast"], labels_c)
P.plot_convrnn_supplement(suppl, before_label="Image on (t=12)", after_label="Final (t=17)")

## Untrained controls (separate supplement)

The random-weight, multi-seed controls live in their own runner because they
load a different activation set (`activations/untrained/`, 5 seeds per model):

```bash
python ../scripts/run_untrained_analysis.py \
    --activations ../activations/untrained \
    --out ../results/untrained --figures ../figures/untrained
```

Or interactively via `repgeo.untrained` — see that module's docstrings.